# Stage 3 — Meeting Insights Generator

**Pipeline:** AMI Dataset → Whisper STT (Stage 2) → **LLM Extraction (Stage 3)**

This stage reads `data/transcripts.json` (output of Stage 2) and uses the Groq API  
to extract structured insights: summary, key points, decisions, and action items.

| | |
|---|---|
| **Input** | `data/transcripts.json` |
| **Output** | `data/stage3_output.json` |
| **Model** | `openai/gpt-oss-120b` via Groq |
| **Prompts** | 2 — Summary/Decisions + Action Items |

## Cell 1 — Imports & Environment Setup

In [5]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import os
import re
import time
from pathlib import Path
from typing import Any
#!pip install groq python-dotenv
# ── Third-party ───────────────────────────────────────────────────────────────
from dotenv import load_dotenv
from groq import Groq

# ── Load .env (GROQ_API_KEY must be defined there) ────────────────────────────
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise EnvironmentError("GROQ_API_KEY not found. Add it to your .env file.")

print("Environment loaded.")
print(f"  API key   : {'*' * 8}{GROQ_API_KEY[-4:]}")

Environment loaded.
  API key   : ********TUVv


## Cell 2 — Configuration

In [6]:
# ── Model & I/O paths ─────────────────────────────────────────────────────────
MODEL            = "openai/gpt-oss-120b"
INPUT_PATH       = Path("data/transcripts.json")
OUTPUT_PATH      = Path("data/stage3_output.json")
MAX_TOKENS       = 1024        # sufficient for structured JSON responses
TEMPERATURE      = 0.0         # deterministic output reduces hallucination
RATE_LIMIT_SLEEP = 2.0         # seconds between API calls (avoid 429s)
MAX_RETRIES      = 3           # retry count on transient errors

# ── Groq client ───────────────────────────────────────────────────────────────
client = Groq(api_key=GROQ_API_KEY)

print(f"Model      : {MODEL}")
print(f"Input      : {INPUT_PATH}")
print(f"Output     : {OUTPUT_PATH}")
print(f"Temperature: {TEMPERATURE}")

Model      : openai/gpt-oss-120b
Input      : data/transcripts.json
Output     : data/stage3_output.json
Temperature: 0.0


## Cell 3 — Prompt Templates

Two separate prompts are used to keep each LLM call focused and reduce confusion:
- **Prompt 1**: Extracts high-level summary, key discussion points, and decisions.
- **Prompt 2**: Extracts concrete action items with owner and deadline.

Design choices to reduce hallucination:
- `temperature=0.0` — no randomness in generation
- Explicit JSON schema embedded in the prompt
- "Only use information from the transcript" instruction
- `null` explicitly allowed for missing fields
- "Do not invent" instructions on every field that could be fabricated

In [7]:
SYSTEM_PROMPT = (
    "You are a precise meeting analyst. "
    "You extract structured information ONLY from the transcript provided. "
    "You never invent, infer, or assume information that is not explicitly stated. "
    "You always respond with valid JSON only — no preamble, no explanation, no markdown fences."
)


def build_summary_prompt(transcript: str) -> str:
    """
    Prompt 1: Extract summary, key discussion points, and decisions.

    Returns a strict JSON string conforming to:
        {
          "summary": "<2-4 sentence overview>",
          "key_points": ["<point>", ...],
          "decisions": ["<decision>", ...]
        }
    """
    return f"""Analyze the meeting transcript below.

Return ONLY a JSON object with this exact structure:
{{
  "summary": "<2-4 sentence factual overview of what was discussed>",
  "key_points": ["<topic or issue raised>", "..."],
  "decisions": ["<explicit decision made>", "..."]
}}

Rules:
- Use ONLY information stated in the transcript. Do not infer or invent.
- If no decisions were made, return an empty list for "decisions".
- If the transcript is empty or unintelligible, return empty strings and empty lists.
- Return valid JSON only. No markdown, no explanation.

TRANSCRIPT:
---
{transcript}
---"""


def build_actions_prompt(transcript: str) -> str:
    """
    Prompt 2: Extract action items with task, owner, and deadline.

    Returns a strict JSON string conforming to:
        {
          "action_items": [
            {"task": "...", "owner": "...", "deadline": "..."}
          ]
        }
    """
    return f"""Extract action items from the meeting transcript below.

Return ONLY a JSON object with this exact structure:
{{
  "action_items": [
    {{
      "task": "<specific task to be done>",
      "owner": "<person responsible, or null if not stated>",
      "deadline": "<deadline or timeframe, or null if not stated>"
    }}
  ]
}}

Rules:
- Extract ONLY tasks explicitly mentioned in the transcript.
- Use null (not the string "null") when owner or deadline is not mentioned.
- If no action items exist, return: {{"action_items": []}}
- Return valid JSON only. No markdown, no explanation.

TRANSCRIPT:
---
{transcript}
---"""


print("Prompt templates defined.")

Prompt templates defined.


## Cell 4 — Core Utilities: LLM Call & JSON Parser

In [8]:
def call_llm(user_prompt: str, retries: int = MAX_RETRIES) -> str:
    """
    Send a prompt to the Groq LLM and return the raw text response.

    Retries on transient errors with exponential backoff.
    Returns an empty string on permanent failure.
    """
    for attempt in range(1, retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_prompt},
                ],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
            )
            return response.choices[0].message.content.strip()

        except Exception as exc:
            wait = 2 ** attempt
            print(f"    [attempt {attempt}/{retries}] Error: {exc}. Retrying in {wait}s...")
            time.sleep(wait)

    print("    [ERROR] All retries exhausted. Returning empty string.")
    return ""


def safe_parse_json(raw: str, fallback: dict) -> dict:
    """
    Attempt to parse a JSON string returned by the LLM.

    Handles two common failure modes:
      1. Model wraps JSON in markdown code fences (```json ... ```)
      2. Model returns completely invalid / empty output

    Returns `fallback` dict on any parse failure.
    """
    if not raw:
        return fallback

    # Strip markdown code fences if present
    cleaned = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned).strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as exc:
        print(f"    [WARN] JSON parse failed: {exc}")
        print(f"    [WARN] Raw response (first 200 chars): {raw[:200]}")
        return fallback


# Fallback structures returned when parsing fails — keeps output schema consistent
SUMMARY_FALLBACK = {"summary": "", "key_points": [], "decisions": []}
ACTIONS_FALLBACK = {"action_items": []}

print("Utilities defined.")

Utilities defined.


## Cell 5 — Load Input Transcripts

In [10]:
# Validate that the input file exists before proceeding
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_PATH}\n"
        "Ensure Stage 2 has completed and saved transcripts.json."
    )

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    transcripts: list[dict] = json.load(f)

# Normalise: Stage 2 may use 'id' or 'sample_id' — handle both
for entry in transcripts:
    if "id" not in entry and "sample_id" in entry:
        entry["id"] = entry["sample_id"]

print(f"Loaded {len(transcripts)} transcript(s) from {INPUT_PATH}")
print()
print("Sample entry keys:", list(transcripts[0].keys()) if transcripts else "(empty)")

Loaded 20 transcript(s) from data/transcripts.json

Sample entry keys: ['sample_id', 'audio_path', 'duration_seconds', 'speaker_id', 'meeting_id', 'begin_time', 'end_time', 'ground_truth_text', 'transcript', 'model', 'id']


## Cell 6 — Main Processing Loop

In [11]:
results: list[dict[str, Any]] = []

total = len(transcripts)

for idx, entry in enumerate(transcripts, start=1):
    sample_id  = entry.get("id", f"unknown_{idx}")
    transcript = entry.get("transcript", "").strip()

    print(f"[{idx}/{total}] Processing sample: {sample_id}")

    # ── Guard: skip empty transcripts gracefully ───────────────────────────────
    if not transcript:
        print(f"  [SKIP] Empty transcript — inserting fallback record.")
        results.append({
            "id":      sample_id,
            "summary": SUMMARY_FALLBACK,
            "actions": ACTIONS_FALLBACK,
        })
        continue

    # ── Prompt 1: Summary + Key Points + Decisions ─────────────────────────────
    print(f"  → Requesting summary & decisions...")
    raw_summary = call_llm(build_summary_prompt(transcript))
    summary_data = safe_parse_json(raw_summary, SUMMARY_FALLBACK)

    time.sleep(RATE_LIMIT_SLEEP)  # respect Groq rate limits between calls

    # ── Prompt 2: Action Item Extraction ──────────────────────────────────────
    print(f"  → Requesting action items...")
    raw_actions = call_llm(build_actions_prompt(transcript))
    actions_data = safe_parse_json(raw_actions, ACTIONS_FALLBACK)

    time.sleep(RATE_LIMIT_SLEEP)  # rate limit between samples

    # ── Assemble output record ─────────────────────────────────────────────────
    record = {
        "id":      sample_id,
        "summary": summary_data,
        "actions": actions_data,
    }
    results.append(record)

    # Preview first action item (if any) for quick validation
    n_actions = len(actions_data.get("action_items", []))
    print(f"  Done. Key points: {len(summary_data.get('key_points', []))}, "
          f"Decisions: {len(summary_data.get('decisions', []))}, "
          f"Action items: {n_actions}")
    print()

print("=" * 60)
print(f"Processing complete. {len(results)} record(s) generated.")

[1/20] Processing sample: EN2001a_clip00
  → Requesting summary & decisions...
  → Requesting action items...
  Done. Key points: 1, Decisions: 0, Action items: 0

[2/20] Processing sample: EN2001a_clip01
  → Requesting summary & decisions...
  → Requesting action items...
  Done. Key points: 1, Decisions: 0, Action items: 0

[3/20] Processing sample: EN2001b_clip00
  → Requesting summary & decisions...
  → Requesting action items...
  Done. Key points: 1, Decisions: 0, Action items: 0

[4/20] Processing sample: EN2001b_clip01
  → Requesting summary & decisions...
  → Requesting action items...
  Done. Key points: 2, Decisions: 0, Action items: 0

[5/20] Processing sample: EN2001d_clip00
  → Requesting summary & decisions...
  → Requesting action items...
  Done. Key points: 0, Decisions: 0, Action items: 0

[6/20] Processing sample: EN2001d_clip01
  → Requesting summary & decisions...
  → Requesting action items...
  Done. Key points: 1, Decisions: 0, Action items: 0

[7/20] Processin

## Cell 7 — Save Output

In [12]:
# Ensure output directory exists
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Output saved to: {OUTPUT_PATH}")
print(f"Records written: {len(results)}")

Output saved to: data/stage3_output.json
Records written: 20


## Cell 8 — Validation & Quick Preview

In [13]:
# Re-load from disk to verify write integrity
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    verification = json.load(f)

print(f"Verification: {len(verification)} record(s) readable from disk.")
print()

# Print a formatted preview of the first record
if verification:
    first = verification[0]
    print("=" * 60)
    print(f"SAMPLE PREVIEW — id: {first['id']}")
    print("=" * 60)

    summary = first.get("summary", {})
    print(f"\nSUMMARY:\n  {summary.get('summary', '')}")

    print("\nKEY POINTS:")
    for pt in summary.get("key_points", []):
        print(f"  - {pt}")

    print("\nDECISIONS:")
    decisions = summary.get("decisions", [])
    if decisions:
        for d in decisions:
            print(f"  - {d}")
    else:
        print("  (none)")

    print("\nACTION ITEMS:")
    action_items = first.get("actions", {}).get("action_items", [])
    if action_items:
        for item in action_items:
            print(f"  Task    : {item.get('task')}")
            print(f"  Owner   : {item.get('owner')}")
            print(f"  Deadline: {item.get('deadline')}")
            print()
    else:
        print("  (none)")

    print()
    print("Stage 3 complete. Proceed to Stage 4.")

Verification: 20 record(s) readable from disk.

SAMPLE PREVIEW — id: EN2001a_clip00

SUMMARY:
  The speaker mentions that when you SSH into the system, there is a prominent warning about not performing any actions on the gateway machine.

KEY POINTS:
  - Warning about doing nothing on the gateway machine when SSHing

DECISIONS:
  (none)

ACTION ITEMS:
  (none)

Stage 3 complete. Proceed to Stage 4.
